# Task 4: Neo4j Kafka Sink Connector & Replay-Safe Graph Ingestion

Tài liệu này ghi nhận quá trình cấu hình, kiểm thử, và vận hành Task 4 trong dự án Nhập môn Dữ liệu lớn.

## 1. Mục tiêu & Các Ràng Buộc Thiết Kế
- **Replay-Safe (Idempotency)**: Đảm bảo khi phát lại (replay) các sự kiện, đồ thị không bị nhân đôi (duplicate) và không bị sai lệch cấu trúc.
- **Stale Event Policy**: Các sự kiện lỗi thời (stale events) do mạng chậm hoặc replay không được ghi đè lên trạng thái mới của đồ thị.
- **Edge-Before-Node Handling**: Khi một quan hệ (Edge) được ghi nhận trước nút nguồn/đích của nó, connector tự động tạo các nút placeholder để giữ tính toàn vẹn tham chiếu đồ thị.
- **Dead Letter Queue (DLQ)**: Các record lỗi cú pháp hoặc hỏng schema được chuyển hướng đến topic `connector.errors` mà không làm ngắt quãng pipeline.
- **Compose Isolation**: Neo4j và Kafka Connect chạy độc lập trong `infra/docker-compose.neo4j.yml` để không ảnh hưởng đến base Compose.

## 2. Thiết Lập Môi Trường

In [1]:
import os
import subprocess
import json
from pathlib import Path

PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)
print("PROJECT_ROOT:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming


## 3. Kiểm Tra Trạng Thái Connector Đang Hoạt Động

In [2]:
# Xem danh sách connector đang chạy
res = subprocess.run(["curl", "-s", "http://localhost:8083/connectors"], capture_output=True, text=True, check=True)
print("Active Connectors:", json.loads(res.stdout))

Active Connectors: ['neo4j-nodes-sink', 'neo4j-edges-sink']


## 4. Kiểm Tra Chi Tiết Cấu Hình Nodes Sink Connector

In [3]:
res = subprocess.run(["curl", "-s", "http://localhost:8083/connectors/neo4j-nodes-sink/config"], capture_output=True, text=True, check=True)
config = json.loads(res.stdout)
print(json.dumps(config, indent=2))

{
  "connector.class": "streams.kafka.connect.sink.Neo4jSinkConnector",
  "neo4j.authentication.basic.password": "CHANGE_ME_NEO4J_PASSWORD",
  "topics": "cpg.nodes",
  "neo4j.topic.cypher.cpg.nodes": "CALL { WITH event WITH event WHERE event.event_type = 'NODE_UPSERT' MERGE (n:CPGNode {id: event.node.node_id}) ON CREATE SET n += event.node.properties, n.placeholder = false, n.node_type = event.node.node_type, n.name = event.node.name, n.qualified_name = event.node.qualified_name, n.ast_path = event.node.ast_path, n.line_start = event.node.line_start, n.column_start = event.node.column_start, n.line_end = event.node.line_end, n.column_end = event.node.column_end, n.repository_id = event.repository_id, n.commit_sha = event.commit_sha, n.file_path = event.file_path, n.file_id = event.file_id, n.parser_version = event.parser_version, n.schema_version = event.schema_version, n.content_hash = event.content_hash ON MATCH SET n += CASE WHEN n.content_hash IS NULL OR n.content_hash <> event.con

## 5. Kiểm Tra Chi Tiết Cấu Hình Edges Sink Connector

In [4]:
res = subprocess.run(["curl", "-s", "http://localhost:8083/connectors/neo4j-edges-sink/config"], capture_output=True, text=True, check=True)
config = json.loads(res.stdout)
print(json.dumps(config, indent=2))

{
  "connector.class": "streams.kafka.connect.sink.Neo4jSinkConnector",
  "neo4j.topic.cypher.cpg.edges": "CALL { WITH event WITH event WHERE event.event_type = 'EDGE_UPSERT' MERGE (src:CPGNode {id: event.edge.source_id}) ON CREATE SET src.placeholder = true, src.file_id = event.file_id MERGE (dst:CPGNode {id: event.edge.target_id}) ON CREATE SET dst.placeholder = true, dst.file_id = event.file_id MERGE (src)-[r:CPG_EDGE {edge_id: event.edge.edge_id}]->(dst) SET r += CASE WHEN r.content_hash IS NULL OR r.content_hash <> event.content_hash THEN event.edge.properties ELSE {} END, r.edge_type = CASE WHEN r.content_hash IS NULL OR r.content_hash <> event.content_hash THEN event.edge.edge_type ELSE r.edge_type END, r.repository_id = CASE WHEN r.content_hash IS NULL OR r.content_hash <> event.content_hash THEN event.repository_id ELSE r.repository_id END, r.file_id = CASE WHEN r.content_hash IS NULL OR r.content_hash <> event.content_hash THEN event.file_id ELSE r.file_id END, r.parser_versi

## 6. Chạy Script Inspect Kiểm Tra Số Lượng Nodes & Edges Thực Tế Trong Đồ Thị

In [5]:
res = subprocess.run(["python", "scripts/inspect_neo4j_graph.py"], capture_output=True, text=True, check=True)
print(json.dumps(json.loads(res.stdout), indent=2))

{
  "node_count_by_file": [
    {
      "file_id, node_count": "\"test_file_id_edge_ingest\", 2"
    },
    {
      "file_id, node_count": "\"test_file_id_stale_del\", 1"
    },
    {
      "file_id, node_count": "\"dlq_test_key\", 1"
    }
  ],
  "relationship_count_by_file": [],
  "duplicate_nodes": [],
  "duplicate_edges": [],
  "placeholders": [
    {
      "node_id, file_id": "\"test_dst_node_id\", \"test_file_id_edge_ingest\""
    }
  ],
  "content_generations": [
    {
      "file_id, content_hash": "\"test_file_id_edge_ingest\", \"edge_generation_1\""
    },
    {
      "file_id, content_hash": "\"test_file_id_edge_ingest\", NULL"
    },
    {
      "file_id, content_hash": "\"test_file_id_stale_del\", \"generation_new\""
    },
    {
      "file_id, content_hash": "\"dlq_test_key\", \"dlq_verif\""
    }
  ],
  "stale_entities": [],
  "sample_relationships": []
}


## 7. Giải Thích Thuật Toán Trích Xuất & Xử Lý Replay-Safe
- **Subquery (`CALL { ... }`)**: Neo4j 5.x không cho phép lệnh `MATCH` lồng trong `FOREACH`. Thay vào đó, chúng tôi bọc từng khối xử lý (`NODE_UPSERT`/`NODE_DELETE`) vào một khối `CALL { ... }` và lọc bằng `WITH event WHERE event.event_type = '...'`.
- **Map Merge (`+=`)**: Do đồ thị Neo4j không cho phép lưu map lồng trực tiếp trên node, chúng tôi sử dụng toán tử `n += map` để unpack các dynamic properties dạng phẳng lên node, tránh lỗi `Map{}` của driver.
- **Generation Guard**: Việc so sánh `n.content_hash IS NULL OR n.content_hash <> event.content_hash` trước khi gán các thuộc tính đảm bảo các cập nhật muộn (hoặc stale replay) không ghi đè dữ liệu mới.